<a href="https://colab.research.google.com/github/schmitfe/Nawrot_CNS_Course/blob/main/Python_Programming_Day3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercises - Solutions

1. Two arrays below show the maximum and minimum temperature of Cologne in 2018 (`0` corresponds to January, `1` to February, `2` to March, and so on). What is the average temperature for each month? Put the average temperatures into a new array called `mean_temp`. Then plot `mean_temp` versus `months`, where `months=['January', 'February', 'March', ..., 'December']`.
```
max_temp = [5,7,11,15,19,22,24,24,20,15,10,6]
min_temp = [-1,-1,2,4,8,11,13,13,10,7,3,0]
```


In [ ]:

from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/schmitfe/Nawrot_CNS_Course.git"
REPO_NAME = "Nawrot_CNS_Course"


def prepare_repo() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "data").is_dir():
        return cwd

    clone_parent = Path("/content") if "google.colab" in sys.modules else cwd
    repo_root = clone_parent / REPO_NAME
    if not repo_root.exists():
        subprocess.run(["git", "clone", REPO_URL, str(repo_root)], check=True)
    os.chdir(repo_root)
    return repo_root.resolve()


REPO_ROOT = prepare_repo()
DATA_DIR = REPO_ROOT / "data" / "python_programming"
print(f"REPO_ROOT = {REPO_ROOT}")
print(f"DATA_DIR = {DATA_DIR}")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

max_temp = np.array([5, 7, 11, 15, 19, 22, 24, 24, 20, 15, 10, 6])
min_temp = np.array([-1, -1, 2, 4, 8, 11, 13, 13, 10, 7, 3, 0])
months = [
    "January", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December",
]
mean_temp = (max_temp + min_temp) / 2

plt.figure(figsize=(10, 4))
plt.plot(months, mean_temp, "o-")
plt.ylabel("mean temperature (degrees C)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


--------------------------------

2. Load the file `DATA_DIR / "CarData.mat"`. This is a MATLAB `.mat` file, so use `loadmat` from `scipy.io`.

Read the string stored in the key `Description`. How many dimensions does the value stored in the key `M` have? Not all measurements are available for all cars. Missing values are represented by `NaN` (Not a Number), which marks entries that should not be treated as ordinary numerical observations.

2.1. Why is it useful to have a dedicated missing-value marker such as `NaN` in a programming language for quantitative science? Discuss with your partner.

2.2. How many different numbers of cylinders do these cars have? Explore and use the NumPy function `unique`.

2.3. Find the indices of all cars that have 4 or fewer cylinders and assign them to the variable `idx_4`. How much do these cars weigh on average? What is their average fuel economy in miles per gallon? Careful: if you average over `NaN` values, your result will also become `NaN`. Use `np.isnan` (or its negation) to identify missing entries. You may then use `np.nanmean` to compute means while ignoring missing values. Why can this still be dangerous if you do not first inspect where the `NaN` values occur?

2.4. Find the indices of all cars that have 6 or more cylinders and assign them to the variable `idx_6`. Again determine the average weight of those cars. Should `idx_4` and `idx_6` contain any identical indices? To check this you can use `np.intersect1d`. Which average weight is larger?

2.5. Test for a significant difference in weight between cars with 4 or fewer cylinders and cars with 6 or more cylinders using a rank-based two-sample test such as `scipy.stats.ranksums` or `scipy.stats.mannwhitneyu`.

2.6. Now consider all cars again. Do you expect that car weight correlates with fuel economy? Compute the linear correlation coefficient, for example with `np.corrcoef`, and report the significance with `scipy.stats.pearsonr`. As a non-parametric alternative, compute Spearman's rank correlation with `scipy.stats.spearmanr`.


In [ ]:
from scipy.io import loadmat
from scipy.stats import mannwhitneyu, pearsonr, spearmanr

car_data = loadmat(DATA_DIR / "CarData.mat", squeeze_me=True)
print(car_data["Description"])
M = car_data["M"]
print("M dimensions:", M.ndim)

cylinders = M[:, 0]
fuel_economy = M[:, 3]
weight = M[:, 4]
print("cylinder values:", np.unique(cylinders[~np.isnan(cylinders)]))

idx_4 = np.where(cylinders <= 4)[0]
idx_6 = np.where(cylinders >= 6)[0]
print("<= 4 cylinders: mean weight =", np.nanmean(weight[idx_4]),
      "mean fuel economy (mpg) =", np.nanmean(fuel_economy[idx_4]))
print(">= 6 cylinders: mean weight =", np.nanmean(weight[idx_6]))
print("overlap:", np.intersect1d(idx_4, idx_6))

valid_groups = ~np.isnan(weight)
test = mannwhitneyu(
    weight[idx_4][valid_groups[idx_4]],
    weight[idx_6][valid_groups[idx_6]],
    alternative="two-sided",
)
print("weight difference:", test)

valid = ~np.isnan(weight) & ~np.isnan(fuel_economy)
print("Pearson:", pearsonr(weight[valid], fuel_economy[valid]))
print("Spearman:", spearmanr(weight[valid], fuel_economy[valid]))


-------